# 16.7 精度能降低嗎？

# 第 16 章：由量測帶出效率 BKM

前置：已有第4章能forward與生成的模型。像整理廚房，先找哪一步最慢再改工作流程。CPU小例只驗證數值與資料契約；GPU速度、峯值記憶體與編譯收益必須在讀者的硬體實測。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：舊筆記不用重寫：只加新字的K/V**

cache保存每層筆記；不是提前知道下一字答案。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/cache.svg")))

**先想一想：**佔同樣bytes就代表可表示的範圍一樣嗎？

用較短的浮點表示可省空間與加快某些硬體，但動態範圍與精度也改了。autocast替適合運算選dtype，loss等敏感量常保留FP32。

**把直覺寫成數學：**FP16與BF16都16bits，但exponent/mantissa配置不同；訓練FP16常需GradScaler。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
a = torch.randn(4, 4)
b = torch.randn(4, 4)
with torch.autocast("cpu", dtype=torch.bfloat16):
    low = a @ b
full = a @ b
print(low.dtype, "誤差", (low.float() - full).abs().max().item())

**如何讀結果：**這是CPU BF16數值例，不能證明CUDA Tensor Core加速。MPS/CUDA依支持選dtype並測。

**只改一件事：**只把輸入幅度放大，觀察數值風險。
